# CFTR mRNA Dependency Map

Compute nucleotide dependency maps on the CFTR mature mRNA (ORF, ~4443 bp) to test
whether genomic foundation models capture long-range mRNA folding interactions.

**Three CFTR folding pairs** (all share variant 2 at chr7:117,595,001 T>G):
1. `CFTR:7:117509093:G:A:P | CFTR:7:117595001:T:G:P` — ~86 kb apart (genomic)
2. `CFTR:7:117594991:G:T:P | CFTR:7:117595001:T:G:P` — ~10 bp apart (genomic)
3. `CFTR:7:117592008:A:G:P | CFTR:7:117595001:T:G:P` — ~3 kb apart (genomic)

After intron removal these map to positions within the ~4.4 kb ORF.

**Models tested:**
- SpeciesLM metazoa (8 k context, k=6 6-mers, `homo_sapiens` proxy)
- HyenaDNA (60 k context, character-level)

**Sections:**
1. Setup & coordinate mapping
2. Full-ORF dependency maps
3. Windowed dependency maps around each folding pair
4. Epistasis geometry from pre-computed embeddings
5. Summary comparison

## 1. Setup & coordinate mapping

In [ ]:
import sys
from pathlib import Path

# Ensure genebeddings is importable
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "genebeddings").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

from seqmat.gene import Gene

from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import (
    add_epistasis_metrics,
    compute_logodds_dependency_map,
    compute_embedding_perturbation_map,
    compute_nucleotide_dependency_map,
)

In [ ]:
# --- Load CFTR mature mRNA (ORF) ---
orf = Gene.from_file('CFTR').transcript().generate_pre_mrna().generate_mature_mrna().orf
seq = orf.seq
genomic_index = orf.index  # array mapping each mRNA position -> genomic coordinate

print(f"CFTR ORF length: {len(seq)} bp")
print(f"Genomic coordinate range: {genomic_index[0]:,} – {genomic_index[-1]:,}")

# --- CFTR folding pairs (genomic coordinates) ---
CFTR_PAIRS = [
    {
        'name': 'Pair 1 (86 kb)',
        'epi_id': 'CFTR:7:117509093:G:A:P|CFTR:7:117595001:T:G:P',
        'pos1_genomic': 117509093,
        'pos2_genomic': 117595001,
    },
    {
        'name': 'Pair 2 (10 bp)',
        'epi_id': 'CFTR:7:117594991:G:T:P|CFTR:7:117595001:T:G:P',
        'pos1_genomic': 117594991,
        'pos2_genomic': 117595001,
    },
    {
        'name': 'Pair 3 (3 kb)',
        'epi_id': 'CFTR:7:117592008:A:G:P|CFTR:7:117595001:T:G:P',
        'pos1_genomic': 117592008,
        'pos2_genomic': 117595001,
    },
]

# --- Map genomic -> mRNA positions ---
genomic_to_mrna = {int(g): i for i, g in enumerate(genomic_index)}

for pair in CFTR_PAIRS:
    p1 = genomic_to_mrna.get(pair['pos1_genomic'])
    p2 = genomic_to_mrna.get(pair['pos2_genomic'])
    pair['pos1_mrna'] = p1
    pair['pos2_mrna'] = p2
    dist_mrna = abs(p2 - p1) if (p1 is not None and p2 is not None) else None
    pair['dist_mrna'] = dist_mrna
    print(f"{pair['name']}: genomic {pair['pos1_genomic']:,} / {pair['pos2_genomic']:,}"
          f"  →  mRNA pos {p1} / {p2}  (Δ = {dist_mrna} bp)")

# Verify all positions mapped successfully
assert all(p['pos1_mrna'] is not None and p['pos2_mrna'] is not None for p in CFTR_PAIRS), \
    "Some genomic positions did not map to the ORF — check variant coordinates"

## 2. Windowed dependency maps around each folding pair

Running the full ORF (~4.4 k positions × 3 mutations = ~13 k forward passes) is expensive.
Instead, for each folding pair we extract a window that spans both variant positions
plus padding, and compute the dependency map within that window.

Models:
- **SpeciesLM metazoa** — 8 k context, k=6 6-mers, `homo_sapiens` proxy
- **HyenaDNA** — 60 k context, character-level

Methods:
- `compute_nucleotide_dependency_map` — paper's unmasked logit-difference (requires MLM head)
- `compute_embedding_perturbation_map` — cosine distance on token embeddings (any model)

In [ ]:
from genebeddings.wrappers import SpeciesLMWrapper, HyenaDNAWrapper

# --- Model configurations ---
MODELS = [
    ('SpeciesLM (metazoa)', SpeciesLMWrapper, {'preset': 'metazoa'}, 8192,
     {'species_proxy': 'homo_sapiens'}),
    ('HyenaDNA', HyenaDNAWrapper, {}, 60_000, {}),
]

# --- Window configuration ---
PAD = 200          # bp on each side beyond the two variant positions
MAX_POSITIONS = 500  # cap positions per window to keep runtime reasonable

In [ ]:
# --- Compute dependency maps for each pair × model × method ---
results = {}  # key: (pair_name, model_name, method) -> DependencyMapResult

for pair in CFTR_PAIRS:
    p1 = pair['pos1_mrna']
    p2 = pair['pos2_mrna']
    lo = max(0, min(p1, p2) - PAD)
    hi = min(len(seq), max(p1, p2) + PAD + 1)
    window_seq = seq[lo:hi]
    window_len = hi - lo

    # Auto-step: keep positions ≤ MAX_POSITIONS while always including variant sites
    step = max(1, window_len // MAX_POSITIONS)

    # Build position list: regular grid + ensure both variant sites are included
    p1_win = p1 - lo
    p2_win = p2 - lo
    grid_positions = set(range(0, window_len, step))
    grid_positions.update([p1_win, p2_win])
    positions = sorted(grid_positions)

    # Store for plotting
    pair['win_lo'] = lo
    pair['win_hi'] = hi
    pair['p1_win'] = p1_win
    pair['p2_win'] = p2_win
    pair['positions'] = positions
    # Index of variant sites within the subsampled position list
    pair['p1_idx'] = positions.index(p1_win)
    pair['p2_idx'] = positions.index(p2_win)

    print(f"\n{'='*70}")
    print(f"{pair['name']}: mRNA window [{lo}, {hi}) = {window_len} bp, "
          f"step={step}, {len(positions)} positions")
    print(f"  Variant 1 at win pos {p1_win} (idx {pair['p1_idx']}), "
          f"Variant 2 at win pos {p2_win} (idx {pair['p2_idx']})")

    for model_name, model_cls, model_init, max_ctx, model_kwargs in MODELS:
        if window_len > max_ctx:
            print(f"  Skipping {model_name}: window ({window_len} bp) exceeds context ({max_ctx})")
            continue

        print(f"\n  Running: {model_name}")
        model = model_cls(**model_init)

        # --- Nucleotide dependency map (paper's method) ---
        if hasattr(model, 'predict_nucleotides'):
            print(f"    → nucleotide dependency map (unmasked logit-diff), {len(positions)} positions")
            result_nuc = compute_nucleotide_dependency_map(
                model, window_seq, positions=positions,
                show_progress=True, **model_kwargs,
            )
            results[(pair['name'], model_name, 'nuc-dependency')] = result_nuc

            score = result_nuc.matrix[pair['p1_idx'], pair['p2_idx']]
            print(f"    Score at pair: {score:.4f}")

        # --- Embedding perturbation map ---
        print(f"    → embedding perturbation map (cosine), {len(positions)} positions")
        result_embed = compute_embedding_perturbation_map(
            model, window_seq, positions=positions,
            diff="cosine", show_progress=True, **model_kwargs,
        )
        results[(pair['name'], model_name, 'embed-perturb')] = result_embed

        score = result_embed.matrix[pair['p1_idx'], pair['p2_idx']]
        print(f"    Score at pair: {score:.4f}")

        del model

print("\nDone — all dependency maps computed.")

## 3. Visualize dependency maps with folding pair overlay

In [ ]:
# --- Heatmaps for each pair, with variant positions marked ---
for pair in CFTR_PAIRS:
    pair_name = pair['name']
    p1_idx = pair['p1_idx']
    p2_idx = pair['p2_idx']

    # Collect all results for this pair
    pair_results = {(mn, meth): res for (pn, mn, meth), res in results.items() if pn == pair_name}
    if not pair_results:
        print(f"No results for {pair_name}")
        continue

    n_plots = len(pair_results)
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5), squeeze=False)

    for idx, ((model_name, method), result) in enumerate(pair_results.items()):
        ax = axes[0, idx]
        mat = result.matrix.copy()
        mat[np.isnan(mat)] = 0
        np.fill_diagonal(mat, 0)

        im = ax.imshow(mat, cmap='magma', origin='upper', aspect='equal')

        # Mark the folding pair (using subsampled indices)
        ax.scatter(p2_idx, p1_idx, s=120, facecolors='none', edgecolors='lime',
                   linewidths=2, zorder=5, label='Folding pair')
        ax.scatter(p1_idx, p2_idx, s=120, facecolors='none', edgecolors='lime',
                   linewidths=2, zorder=5)

        score = mat[p1_idx, p2_idx]
        # Percentile rank of this score
        upper_tri = mat[np.triu_indices_from(mat, k=1)]
        pctile = 100 * (upper_tri < score).sum() / len(upper_tri) if len(upper_tri) > 0 else 0

        ax.set_title(f"{model_name}\n{method}\nscore={score:.3f} ({pctile:.0f}th pctile)", fontsize=10)
        ax.set_xlabel("Position index (subsampled)")
        ax.set_ylabel("Position index (subsampled)")
        fig.colorbar(im, ax=ax, shrink=0.7)

    fig.suptitle(f"{pair_name}\nepi_id: {pair['epi_id']}", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Epistasis geometry from pre-computed embeddings

Load pre-computed embeddings from the `mature_embeddings/` databases and compute
epistasis metrics for the 3 CFTR folding pairs and their neighborhoods.

In [ ]:
# --- Pre-computed model databases ---
MODEL_DBS = [
    ('ConvNova',       'mature_embeddings/convnova.db'),
    ('MutBERT',        'mature_embeddings/mutbert.db'),
    ('HyenaDNA',       'mature_embeddings/hyenadna.db'),
    ('NT (500m)',      'mature_embeddings/nt500_multi.db'),
    ('Borzoi',         'mature_embeddings/borzoi.db'),
    ('SpeciesLM',      'mature_embeddings/specieslm.db'),
]

# --- Load epistasis metrics for each CFTR pair from each model ---
epi_results = {}  # key: (pair_name, model_name) -> row from metrics df

for pair in CFTR_PAIRS:
    epi_id = pair['epi_id']
    print(f"\n{pair['name']}: {epi_id}")

    for model_name, db_path in MODEL_DBS:
        try:
            db = VariantEmbeddingDB(db_path)
            # Create a single-row dataframe for this pair
            df_single = pd.DataFrame([{'epistasis_id': epi_id}])
            metrics = add_epistasis_metrics(
                df_single, db, model=None, id_col='epistasis_id', show_progress=False,
            )
            db.close()

            if len(metrics) > 0 and not metrics.iloc[0].get('_skipped', False):
                row = metrics.iloc[0]
                epi_results[(pair['name'], model_name)] = row
                print(f"  {model_name}: epi_R_singles={row.get('epi_R_singles', 'N/A'):.4f}, "
                      f"magnitude_ratio={row.get('magnitude_ratio', 'N/A'):.4f}")
            else:
                print(f"  {model_name}: skipped (embeddings not in DB)")
        except Exception as e:
            print(f"  {model_name}: error — {e}")

## 5. Summary: dependency map scores at folding pair positions

Compare the dependency map score at each folding pair against the background
distribution of all pairwise scores in the window.

In [ ]:
# --- Summary table ---
rows = []
for (pair_name, model_name, method), result in results.items():
    pair = next(p for p in CFTR_PAIRS if p['name'] == pair_name)
    mat = result.matrix.copy()
    mat[np.isnan(mat)] = 0
    np.fill_diagonal(mat, 0)

    score = mat[pair['p1_idx'], pair['p2_idx']]
    upper_tri = mat[np.triu_indices_from(mat, k=1)]
    pctile = 100 * (upper_tri < score).sum() / len(upper_tri)
    mean_bg = upper_tri.mean()
    std_bg = upper_tri.std()
    z_score = (score - mean_bg) / std_bg if std_bg > 0 else 0

    rows.append({
        'Pair': pair_name,
        'Model': model_name,
        'Method': method,
        'Score': score,
        'Percentile': pctile,
        'Z-score': z_score,
        'BG mean': mean_bg,
        'BG std': std_bg,
    })

df_summary = pd.DataFrame(rows)
display(df_summary.sort_values(['Pair', 'Model', 'Method']).reset_index(drop=True))

# --- Bar chart: percentile rank of each folding pair ---
fig, ax = plt.subplots(figsize=(10, max(3, len(df_summary) * 0.5)))
labels = [f"{r['Pair']} | {r['Model']} | {r['Method']}" for _, r in df_summary.iterrows()]
colors = ['#0072B2' if 'embed' in m else '#D55E00' for m in df_summary['Method']]
ax.barh(range(len(df_summary)), df_summary['Percentile'], color=colors)
ax.set_yticks(range(len(df_summary)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Percentile rank (within window)')
ax.set_title('CFTR folding pair scores vs background')
ax.axvline(95, color='red', linestyle='--', alpha=0.5, label='95th percentile')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Epistasis geometry summary

Compare epistasis metrics for the 3 CFTR folding pairs across all pre-computed models.

In [ ]:
# --- Epistasis metrics table ---
epi_rows = []
for (pair_name, model_name), row in epi_results.items():
    epi_rows.append({
        'Pair': pair_name,
        'Model': model_name,
        'epi_R_singles': row.get('epi_R_singles', np.nan),
        'epi_R_raw': row.get('epi_R_raw', np.nan),
        'magnitude_ratio': row.get('magnitude_ratio', np.nan),
        'cos_sim_expected_observed': row.get('cos_sim_expected_observed', np.nan),
    })

if epi_rows:
    df_epi = pd.DataFrame(epi_rows)
    display(df_epi.sort_values(['Pair', 'Model']).reset_index(drop=True))

    # Heatmap of epi_R_singles: pairs × models
    pivot = df_epi.pivot_table(index='Pair', columns='Model', values='epi_R_singles')
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax, center=0)
    ax.set_title('Epistasis non-additivity (epi_R_singles) — CFTR folding pairs')
    plt.tight_layout()
    plt.show()
else:
    print("No epistasis results found in pre-computed databases.")